In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install polars

In [4]:
import json
import numpy as np
import os
import polars as pl
from typing import Dict, List, Set, Tuple

In [5]:
class Config:
  TRAIN_PATH = "/content/drive/MyDrive/dataset/train.parquet"
  BENCHMARK_QUERIES_PATH = "/content/drive/MyDrive/dataset/benchmark_queries.parquet"
  BENCHMARK_ITEMS_PATH = "/content/drive/MyDrive/dataset/benchmark_items.parquet"

  # Validation split parameters
  VAL_NUM_QUERIES = 2452
  VAL_CORPUS_SIZE = 189212
  VAL_DIR = "/content/drive/MyDrive/dataset/val_split"

  RANDOM_SEED = 42

In [ ]:
train_df = pl.read_parquet(Config.TRAIN_PATH)
bench_q_df = pl.read_parquet(Config.BENCHMARK_QUERIES_PATH)
bench_i_df = pl.read_parquet(Config.BENCHMARK_ITEMS_PATH)

# Validation split

First, we create a validation split in order to be able to assess the quality of the pipeline before submitting.

We'll take 2452 queries and 189 212 items. Based on EDA we know that there are duplicate queries in train set, so when we put a query into a validation set, we take all instances of this query as well.

In order for a validation set to re representative, we want the majority of picked queries to be frequent in train dataset - 70% of our validation set will be formed from those. The rest30 % we form using rarely encountered queries to make the validation set more diversed.

In [ ]:
class ValidationSplitter:

  @staticmethod
  def create_split(train_df: pl.DataFrame):
    os.makedirs(Config.VAL_DIR, exist_ok=True)
    np.random.seed(Config.RANDOM_SEED)

    # Group query, count items for a query and get a list of them
    q_counts = (
        train_df.group_by("search_query")
        .agg(pl.len().alias("clicks_count"), pl.col("item_id").unique().alias("relevant_items"))
      )

    q_sorted = q_counts.sort("clicks_count", descending=True)

    num_frequent = int(Config.VAL_NUM_QUERIES * 0.7)
    num_rare = Config.VAL_NUM_QUERIES - num_frequent

    # Choose frequent and rare queries by sampling out of 20000 most/least frequent ones
    n_queries_to_pool = 20000
    freq_pool = q_sorted.head(n_queries_to_pool)
    rare_pool = q_sorted.tail(q_sorted.height - n_queries_to_pool)

    val_freq_queries = freq_pool.sample(n=num_frequent, seed=Config.RANDOM_SEED)
    val_rare_queries = rare_pool.sample(n=num_rare, seed=Config.RANDOM_SEED)

    # Shuffle chosen queries
    val_selected_q = pl.concat([val_freq_queries, val_rare_queries]).sample(fraction=1.0, shuffle=True, seed=Config.RANDOM_SEED)
    val_query_texts = set(val_selected_q["search_query"].to_list())

    # Take rows with validation queries
    val_data_df = train_df.filter(pl.col("search_query").is_in(val_query_texts))
    # Create train subset removing the validation queries
    train_sub_df = train_df.filter(~pl.col("search_query").is_in(val_query_texts))


    val_queries = (
        val_data_df.select([
            "search_query", "search_location_id", "search_is_delivery_search",
            "search_infm_params_text", "search_category"
        ])
        .unique(subset=["search_query", "search_location_id"]) # filter identical queries from the same location (we'll group their items later)
        .head(Config.VAL_NUM_QUERIES)
        )

    # There are no query_ids in train set, but there are in benchmarks, so we create custom ones for validation as well in HEX
    custom_qids = [f"{i:016x}" for i in range(val_queries.height)]
    val_queries = val_queries.with_columns(pl.Series("query_id", custom_qids))

    # Forming validations set with ground truth
    val_with_gt = val_data_df.join(
            val_queries.select(["query_id", "search_query", "search_location_id"]),
            on=["search_query", "search_location_id"],
            how="inner"
        )

    gt_map = (
            val_with_gt.group_by("query_id")
            .agg(pl.col("item_id").unique())
            .to_dict(as_series=False)
        )

    gt_dict = {qid: items for qid, items in zip(gt_map["query_id"], gt_map["item_id"])}

    # Now we form a corpus of all relevant items for validation
    all_relevant_items = set()
    for items in gt_dict.values():
      all_relevant_items.update(items)

    # Add items features
    relevant_items_df = (
        train_df.filter(pl.col("item_id").is_in(all_relevant_items))
        .select([
            "item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
            "item_category_id", "item_microcat_id", "item_price", "item_rating",
            "item_rating_reviews_count", "item_location_id", "item_latitude",
            "item_longitude", "item_is_phone_hidden", "item_is_message_forbidden"
        ])
        .unique(subset=["item_id"])
    )

    # Add non-relevant items to get to he desired size of the corpus
    left_to_fill = Config.VAL_CORPUS_SIZE - relevant_items_df.height
    negative_items_df = (
        train_df.filter(~pl.col("item_id").is_in(all_relevant_items))
        .select(relevant_items_df.columns)
        .unique(subset=["item_id"])
        .sample(n=left_to_fill, seed=Config.RANDOM_SEED)
    )

    val_corpus_df = pl.concat([relevant_items_df, negative_items_df]).sample(fraction=1.0, shuffle=True, seed=Config.RANDOM_SEED)

    val_queries.write_parquet(os.path.join(Config.VAL_DIR, "val_queries.parquet"))
    val_corpus_df.write_parquet(os.path.join(Config.VAL_DIR, "val_items.parquet"))
    train_sub_df.write_parquet(os.path.join(Config.VAL_DIR, "train_sub.parquet"))

    with open(os.path.join(Config.VAL_DIR, "val_ground_truth.json"), "w", encoding="utf-8") as f:
      json.dump(gt_dict, f, ensure_ascii=False)


Performing the split and saving the new subsets to Google drive

In [ ]:
ValidationSplitter.create_split(train_df)

__________________

# BM25 + Geo Search

As our first method for searching relevant documents based on our queries we use BM25 with additional weights to parts of our textual input. BM25 is a good choice since it's more efficient yet able to yield competitive results, no need for GPU access.

A document in our search engine will be formed by the following item properties: "item_title_raw", "item_description_raw", "item_infm_params_text". We choose them since they contain (with certain caveats pointed below) the most relevant information for our task.

We'll assign weights, 3 to the "item_title_raw" field since this field is expected to concentrate the most crucial information regarding the item (the owner/service provider might want to capture potential customers attention).

To the 'item_infm_params_text" field we assign 2. For the "iten_description_raw" we'll use their shortened version with the weight "1" since descriptions may be incoherent, contain unrelated information and therefore serve as distraction.

We also want to make our search geo-aware, since the user might be looking for an on-site serice, while we might find a suitable service placed far away from them, which is of no use for the user and in case of presenting such item to a user, they might be very disappointed with our service quality and less likely to come back.

In order to account for this we take 200 best candidates based on BM25, then if the user isn't choosing delivery ('search_is_delivery_search' == 0) we provide only local candidates first. If after selecting top local candidates we are unable to form a 50 candidates set, we fill the remaining slots with non-locals items.

If the user is choosing delivery, we don't use geographical limits.

In [6]:
!pip install rank_bm25

In [7]:
import re
import json
from typing import List, Dict, Set
from tqdm.auto import tqdm

import nltk
from nltk.stem.snowball import RussianStemmer
from rank_bm25 import BM25Okapi

In [8]:
class LexicalConfig:
    VAL_DIR = "/content/drive/MyDrive/dataset/val_split"
    VAL_QUERIES_PATH = os.path.join(VAL_DIR, "val_queries.parquet")
    VAL_ITEMS_PATH = os.path.join(VAL_DIR, "val_items.parquet")
    VAL_GT_PATH = os.path.join(VAL_DIR, "val_ground_truth.json")

    BENCHMARK_QUERIES_PATH = "/content/drive/MyDrive/dataset/benchmark_queries.parquet"
    BENCHMARK_ITEMS_PATH = "/content/drive/MyDrive/dataset/benchmark_items.parquet"
    OUTPUT_CSV_PATH = "/content/drive/MyDrive/dataset/answer.csv"

    BM25_TOP_POOL = 200
    FINAL_TOP_K = 50

We use `RussianStemmer` from `snowball` package since it is supposed to operate faster. We need to stem text since there are various word endings possible in Russia for words with related meaning, so without doing so we may drastically limit our ability to find relevant items.

We also remove punctuation from the inputs.

In [9]:
CLEAN_PUNCT_REGEX = re.compile(r'[^а-яa-z0-9\s]')
stemmer = RussianStemmer()

In [10]:
def tokenize_and_stem(text: str) -> List[str]:
    if not text:
        return []
    text = text.lower().replace('ё', 'е')
    text = CLEAN_PUNCT_REGEX.sub(' ', text)
    tokens = text.split()
    stemmed = [stemmer.stem(t) if len(t) > 2 else t for t in tokens]
    return stemmed

Here below we introduce a class to prepare data for future processing

In [11]:
class TextBuilder:
  @staticmethod
  def build_item_documents(items_df: pl.DataFrame) -> List[List[str]]:
    # Get necessary data for forming item documents
    titles = items_df["item_title_raw"].fill_null("").to_list()
    params = items_df["item_infm_params_text"].fill_null("").to_list()
    descs = items_df["item_description_raw"].fill_null("").to_list()

    corpus_token = []
    for title, param, desc in tqdm(zip(titles, params, descs), total=len(titles)):
      short_desc = desc[:150] if desc else ""
      doc_str = f"{title} {title} {title} {param} {param} {short_desc}"
      corpus_token.append(tokenize_and_stem(doc_str))
    return corpus_token


  @staticmethod
  def build_query_tokens(queries_df: pl.DataFrame) -> List[List[str]]:
    q_texts = queries_df["search_query"].fill_null("").to_list()
    q_params = queries_df["search_infm_params_text"].fill_null("").to_list()

    queries_tokens = []
    for q, p in zip(q_texts, q_params):
      full_query = f"{q} {p}".strip()
      tokens = tokenize_and_stem(full_query)
      queries_tokens.append(tokens)
    return queries_tokens

Now we write class for BM25 and Geo search engine

In [12]:
class BM25GeoSearchEngine:
  def __init__(self, items_df: pl.DataFrame, corpus_tokens: List[List[str]]):
    self.items_df = items_df
    self.item_ids = items_df["item_id"].to_numpy()
    self.item_locations = items_df["item_location_id"].to_numpy()

    self.bm25 = BM25Okapi(corpus_tokens, k1=1.5, b=0.75)

  def search_query(
    self,
    query_tokens: List[str],
    search_location_id: int,
    is_delivery: int,
    top_k: int = 50
  ) -> List[str]:
    # Score documents
    scores = self.bm25.get_scores(query_tokens)

    # Retrieve best candidates and sort them by score in descending order
    pool_size = min(LexicalConfig.BM25_TOP_POOL, len(scores))
    top_pool_indices = np.argpartition(scores, -pool_size)[-pool_size:]
    top_pool_indices = top_pool_indices[np.argsort(-scores[top_pool_indices])]

    # if a user picks delivery or we don't know the location, return top-50 of BM25 candidates
    if is_delivery == 1 or search_location_id is None:
      return self.item_ids[top_pool_indices[:top_k]].tolist()

    # Geo filter
    local_candidates = []
    other_candidates = []

    for idx in top_pool_indices:
      it_id = self.item_ids[idx]
      it_loc = self.item_locations[idx]

      if it_loc == search_location_id:
          local_candidates.append(it_id)
      else:
          other_candidates.append(it_id)

    # local candidates pool
    selected = local_candidates[:top_k]

    # Add non-local items if couldn't find 50 local ones
    if len(selected) < top_k:
      needed = top_k - len(selected)
      selected.extend(other_candidates[:needed])

    return selected


Evalutation function for recall@50

In [13]:
def compute_recall_at_50(predictions: Dict[str, List[str]], gt_path: str) -> float:
  with open(gt_path, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

  recalls = []
  for qid, rel_items in ground_truth.items():
    if not rel_items:
        continue
    rel_set = set(rel_items)
    preds = predictions.get(qid, [])[:50]
    hits = len(rel_set.intersection(preds))
    recalls.append(hits / len(rel_set))

  return np.mean(recalls) if recalls else 0.0

Test on validation

In [ ]:
val_queries_df = pl.read_parquet(LexicalConfig.VAL_QUERIES_PATH)
val_items_df = pl.read_parquet(LexicalConfig.VAL_ITEMS_PATH)

In [ ]:
corpus_tokens_for_val = TextBuilder.build_item_documents(val_items_df)
queries_tokens_for_val = TextBuilder.build_query_tokens(val_queries_df)

  0%|          | 0/189212 [00:00<?, ?it/s]

In [ ]:
search_engine_for_val = BM25GeoSearchEngine(val_items_df, corpus_tokens_for_val)

In [ ]:
simple_bm25_preds = {}
qids = val_queries_df["query_id"].to_list()

In [ ]:
for qid, q_toks in zip(qids, queries_tokens_for_val):
  candidates = search_engine_for_val.search_query(q_toks, search_location_id=None, is_delivery=1, top_k=50)
  simple_bm25_preds[qid] = candidates

In [ ]:
score_simple_bm25 = compute_recall_at_50(simple_bm25_preds, LexicalConfig.VAL_GT_PATH)

In [ ]:
score_simple_bm25

np.float64(0.25309559630575773)

________

Now we apply the engine to our benchmark data, but this time we'll also use geographical limitations

In [16]:
bench_queries_df = pl.read_parquet(LexicalConfig.BENCHMARK_QUERIES_PATH)
bench_items_df = pl.read_parquet(LexicalConfig.BENCHMARK_ITEMS_PATH)

In [17]:
corpus_tokens = TextBuilder.build_item_documents(bench_items_df)
queries_tokens = TextBuilder.build_query_tokens(bench_queries_df)

  0%|          | 0/189212 [00:00<?, ?it/s]

In [18]:
search_engine = BM25GeoSearchEngine(bench_items_df, corpus_tokens)

In [19]:
qids = bench_queries_df["query_id"].to_list()
locations = bench_queries_df["search_location_id"].to_list()
deliveries = bench_queries_df["search_is_delivery_search"].fill_null(0).to_list()

In [20]:
predictions = []
for qid, q_toks, loc_id, is_del in tqdm(zip(qids, queries_tokens, locations, deliveries), total=len(qids), desc="Predicting Top-50"):
    top50 = search_engine.search_query(
        query_tokens=q_toks,
        search_location_id=loc_id,
        is_delivery=is_del,
        top_k=LexicalConfig.FINAL_TOP_K
    )
    predictions.append(top50)

Predicting Top-50:   0%|          | 0/2452 [00:00<?, ?it/s]

In [21]:
import pandas as pd

In [22]:
answer = pd.DataFrame({
        'query_id': qids,
        'answer': [' '.join(top50) for top50 in predictions]
    })

In [23]:
answer.to_csv(LexicalConfig.OUTPUT_CSV_PATH, index=False, encoding='utf-8')